# THEMAP hero figure — A map of medicinal-chemistry task space

Single-figure presentation visual for the FS-Mol companion data.

**Inputs** (`make download-fsmol` first): `datasets/fsmol_hardness/{ext_chem,embeddings,FSMol_Eval_ProtoNet}`, `datasets/test/test_proteins.csv`.  
**Output**: `assets/themap_hero_figure.{pdf,png}` (gitignored).

The notebook computes a UMAP layout of GIN-supervised-infomax task centroids (4,938 train + 157 test = 5,095 tasks), colors test tasks by protein family, and highlights five representative target tasks with arrows to their top-3 OTDD-nearest training sources.

In [ ]:
"""Development script for the THEMAP hero figure.

Produces a single composite figure for presentations:
  - Main panel: UMAP landscape of all 5,095 FS-Mol tasks (4,938 train + 157 test)
    in centroid-embedding space (GIN-supervised-infomax). Train tasks shown as
    grey background density; test tasks colored by protein family. Arrows from
    a few highlighted test targets to their top-3 OTDD-nearest train sources.
  - Inset (top-right): scatter of OTDD distance-to-nearest-source vs ProtoNet
    delta_auprc, with Spearman rho annotation.

Run:
    cd notebooks && python _figure_dev.py
"""

from __future__ import annotations

import os
import pickle
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import umap
from matplotlib.patches import FancyArrowPatch
from scipy.spatial import ConvexHull
from scipy.stats import spearmanr

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
# In a notebook, cwd starts at notebooks/. Resolve REPO from that.
REPO = Path.cwd() if Path.cwd().name != "notebooks" else Path.cwd().parent
os.chdir(REPO)
DATA = REPO / "datasets" / "fsmol_hardness"
ASSETS = REPO / "assets"
ASSETS.mkdir(exist_ok=True)

FEATURIZER = "gin_supervised_infomax"  # paper's strongest chemical featurizer
SEED = 42
np.random.seed(SEED)

# ---------------------------------------------------------------------------
# Register Latin Modern Roman (visual equivalent of CMU Serif / Computer
# Modern) from TeXLive so the figure looks paper-quality without needing
# text.usetex. Falls back to the rc default if the fonts are not present.
# ---------------------------------------------------------------------------
from matplotlib import font_manager as _fm  # noqa: E402

_LM_DIR = Path("/home/hfooladi/texlive/2025/texmf-dist/fonts/opentype/public/lm")
if _LM_DIR.is_dir():
    for _name in (
        "lmroman10-regular.otf",
        "lmroman10-bold.otf",
        "lmroman10-italic.otf",
        "lmroman10-bolditalic.otf",
        "lmromancaps10-regular.otf",
    ):
        _f = _LM_DIR / _name
        if _f.exists():
            _fm.fontManager.addfont(str(_f))
    SERIF_FAMILY = "Latin Modern Roman"
else:
    SERIF_FAMILY = "DejaVu Serif"
print(f"Using serif family: {SERIF_FAMILY}")

## 1. Load data

Load the OTDD distance matrix, GIN embeddings, ProtoNet 16-shot results, and protein-family metadata.

In [ ]:
print("[1/6] Loading data...")

with open(DATA / "ext_chem" / f"otdd_{FEATURIZER}.pkl", "rb") as f:
    otdd = pickle.load(f)

train_ids = list(otdd["train_chembl_ids"])
test_ids = list(otdd["test_chembl_ids"])
otdd_mat = otdd["distance_matrices"]  # torch.Tensor [N_train, N_test]
if isinstance(otdd_mat, torch.Tensor):
    otdd_mat = otdd_mat.cpu().numpy()
n_nan = int(np.isnan(otdd_mat).sum())
print(
    f"  OTDD matrix: {otdd_mat.shape}, train={len(train_ids)}, test={len(test_ids)}, "
    f"NaN={n_nan} ({n_nan / otdd_mat.size:.1%})"
)
# Drop test tasks that have zero valid sources (1 such task exists).
valid_test_mask = (~np.isnan(otdd_mat)).any(axis=0)
if (~valid_test_mask).any():
    dropped = [test_ids[i] for i, ok in enumerate(valid_test_mask) if not ok]
    print(f"  Dropping {len(dropped)} test task(s) with no valid OTDD: {dropped}")
    test_ids = [t for t, ok in zip(test_ids, valid_test_mask) if ok]
    otdd_mat = otdd_mat[:, valid_test_mask]

with open(DATA / "embeddings" / f"{FEATURIZER}_train.pkl", "rb") as f:
    emb_train_raw = pickle.load(f)
with open(DATA / "embeddings" / f"{FEATURIZER}_test.pkl", "rb") as f:
    emb_test_raw = pickle.load(f)

protonet = pd.read_csv(
    DATA / "FSMol_Eval_ProtoNet" / "summary" / "ProtoNet_summary_num_train_requested_16.csv"
).set_index("assay")
print(f"  ProtoNet 16-shot: {len(protonet)} tasks")

prot_meta = pd.read_csv(REPO / "datasets" / "test" / "test_proteins.csv").set_index("chembl_id")
print(f"  Protein metadata: {len(prot_meta)} tasks")

## 2. Compute task centroids in 300-D embedding space

One mean embedding vector per task gives us a single point per task to feed into UMAP.

In [ ]:
print("[2/6] Computing centroids...")


def task_centroid(entry: dict) -> np.ndarray:
    emb = entry[FEATURIZER]
    if isinstance(emb, torch.Tensor):
        emb = emb.cpu().numpy()
    return emb.mean(axis=0)


train_centroids = np.stack([task_centroid(emb_train_raw[t]) for t in train_ids])
test_centroids = np.stack([task_centroid(emb_test_raw[t]) for t in test_ids])
print(f"  train centroids: {train_centroids.shape}")
print(f"  test centroids:  {test_centroids.shape}")

# Normalize to unit length so cosine ~ Euclidean post-norm (UMAP-friendly).
train_centroids /= np.linalg.norm(train_centroids, axis=1, keepdims=True) + 1e-12
test_centroids /= np.linalg.norm(test_centroids, axis=1, keepdims=True) + 1e-12

joint = np.vstack([train_centroids, test_centroids])
n_train = len(train_ids)

## 3. UMAP layout

UMAP with very global parameters (n_neighbors=200, min_dist=0.6) produces a single connected blob — no fragmentation.

In [ ]:
print("[3/6] Running UMAP (this takes ~60-120s)...")
reducer = umap.UMAP(
    n_neighbors=200,  # very global → single connected blob
    min_dist=0.6,
    metric="cosine",
    init="spectral",
    random_state=SEED,
    n_jobs=1,
)
xy = reducer.fit_transform(joint)
xy_train, xy_test = xy[:n_train], xy[n_train:]
print(f"  layout: train {xy_train.shape}, test {xy_test.shape}")
print(
    f"  layout extents: x=[{xy[:, 0].min():.1f}, {xy[:, 0].max():.1f}], "
    f"y=[{xy[:, 1].min():.1f}, {xy[:, 1].max():.1f}]"
)

## 4. Colour map by protein family

Curated 8-family palette; long-tail families fall into "other" (light grey).

In [ ]:
print("[4/6] Assigning protein-family colors...")

# Use 'protein_family' (finer than super_family for kinases) and bucket the
# long tail into "other".
families_raw = prot_meta.loc[test_ids, "protein_family"].fillna("other")

# Curated set of families we want highlighted (good visual distinction +
# represent both kinase subfamilies and non-kinases). Anything else → "other".
SHOWN_FAMILIES = [
    "kinase_tk",
    "kinase_cmgc",
    "kinase_ste",
    "kinase_agc",
    "kinase_camk",
    "epigenetic_regulator",
    "transferase",
    "protease_serine",
]
families = families_raw.where(families_raw.isin(SHOWN_FAMILIES), other="other")

palette = {
    "kinase_tk": "#E41A1C",
    "kinase_cmgc": "#377EB8",
    "kinase_ste": "#4DAF4A",
    "kinase_agc": "#984EA3",
    "kinase_camk": "#FF7F00",
    "epigenetic_regulator": "#F781BF",
    "transferase": "#A65628",
    "protease_serine": "#17BECF",
    "other": "#BDBDBD",
}
test_colors = families.map(palette).values

## 5. Pick highlighted target tasks + their top-3 OTDD nearest train sources

Greedy spatial diversification: each pick must be inside the dense region and at least MIN_SEP away from previously-picked targets, so labels and arrows do not overlap.

In [ ]:
print("[5/6] Picking highlighted targets...")

# distance-to-nearest-train per test task (in OTDD space, ignoring NaN)
test_nn_dist = np.nanmin(otdd_mat, axis=0)  # shape [N_test]
nn_train_idx = np.nanargmin(otdd_mat, axis=0)  # shape [N_test]

# Pre-compute top-3 OTDD-nearest train neighbours for *every* test task and
# the mean UMAP distance from each test task to those three sources. This lets
# us require that arrows are visually long when picking targets — a target
# whose three nearest sources are right on top of it in UMAP space would have
# arrows shorter than the marker itself, which defeats the figure.
nn3_all: dict[int, np.ndarray] = {}
arrow_visibility = np.zeros(len(test_ids))
for ti in range(len(test_ids)):
    col = otdd_mat[:, ti].copy()
    col[np.isnan(col)] = np.inf
    top3 = np.argsort(col)[:3]
    nn3_all[ti] = top3
    arrow_visibility[ti] = np.mean([np.linalg.norm(xy_test[ti] - xy_train[s]) for s in top3])

target_picks: list[int] = []
fam_arr = families.values
HIGHLIGHT_FAMILIES = [
    "kinase_cmgc",  # kinase representative
    "transferase",  # non-kinase enzyme
    "epigenetic_regulator",
    "protease_serine",
    "kinase_tk",
]

center = xy_train.mean(axis=0)
std = xy_train.std(axis=0)
dense_radius = 1.5 * std


def in_dense(p: np.ndarray) -> bool:
    return abs(p[0] - center[0]) <= dense_radius[0] and abs(p[1] - center[1]) <= dense_radius[1]


_xrange = xy_train[:, 0].max() - xy_train[:, 0].min()
_yrange = xy_train[:, 1].max() - xy_train[:, 1].min()
diag = np.sqrt(_xrange**2 + _yrange**2)
MIN_SEP = 0.14 * diag
# arrows must reach at least 8% of the diagonal on average, so they read clearly
ARROW_MIN = 0.06 * diag


def far_from_picked(p: np.ndarray) -> bool:
    return all(np.linalg.norm(p - xy_test[j]) >= MIN_SEP for j in target_picks)


for fam in HIGHLIGHT_FAMILIES:
    idxs = np.where(fam_arr == fam)[0]
    if len(idxs) == 0:
        continue
    # Strict: arrows must be visibly long AND target spatially separated AND
    # inside the dense region. If no kinase_camk task (etc.) satisfies this,
    # we SKIP that family rather than show invisible-arrow stars — the figure
    # is honest about which families have nearest sources far in UMAP space.
    pool = [
        int(i)
        for i in idxs
        if in_dense(xy_test[i]) and arrow_visibility[i] >= ARROW_MIN and far_from_picked(xy_test[i])
    ]
    if not pool:
        # one relaxation: allow non-dense (peripheral) targets, still enforce
        # spatial separation and visible arrows
        pool = [int(i) for i in idxs if arrow_visibility[i] >= ARROW_MIN and far_from_picked(xy_test[i])]
    if not pool:
        print(f"  Skipping {fam}: no candidate with visible arrows + separation")
        continue
    pool.sort(key=lambda i: -arrow_visibility[i])
    target_picks.append(pool[0])
    if len(target_picks) >= 3:
        break

print(f"  Highlighted targets ({len(target_picks)}):")
for i in target_picks:
    print(
        f"    {test_ids[i]:<14} fam={fam_arr[i]:<24} "
        f"OTDD_nn={test_nn_dist[i]:.3f}  arrow_len={arrow_visibility[i]:.2f}"
    )

nn3_per_target = {i: nn3_all[i] for i in target_picks}

## 6. Build figure

Single composite map: density backdrop → train scatter → family hulls → test scatter → arrows + halos → labels → title + legend.

In [ ]:
print("[6/6] Rendering figure...")

plt.rcParams.update(
    {
        "font.family": "serif",
        "font.serif": [SERIF_FAMILY, "DejaVu Serif", "serif"],
        "font.size": 12,
        "axes.linewidth": 0.8,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "savefig.bbox": "tight",
        "mathtext.fontset": "cm",  # math (rho, Delta, etc.) in Computer Modern
    }
)

fig = plt.figure(figsize=(14, 9))
ax = fig.add_axes([0.02, 0.04, 0.96, 0.94])
ax.set_xticks([])
ax.set_yticks([])
for s in ax.spines.values():
    s.set_visible(False)
ax.set_facecolor("#fdfdfd")

# Crop tightly to the main dense cluster — UMAP often separates a small ~5%
# satellite of train tasks; cropping to 8th-99th percentile drops it.
# Extra bottom padding so target labels at the canvas edge don't clip.
xlo, xhi = np.percentile(xy_train[:, 0], [8, 99])
ylo, yhi = np.percentile(xy_train[:, 1], [3, 99])
xpad = 0.03 * (xhi - xlo)
ypad_top = 0.02 * (yhi - ylo)
ypad_bot = 0.10 * (yhi - ylo)
ax.set_xlim(xlo - xpad, xhi + xpad)
ax.set_ylim(ylo - ypad_bot, yhi + ypad_top)

# 6a. background density of train tasks
sns.kdeplot(
    x=xy_train[:, 0],
    y=xy_train[:, 1],
    levels=8,
    fill=True,
    cmap="Blues",
    alpha=0.35,
    bw_adjust=0.9,
    ax=ax,
)

# 6b. train scatter (small, translucent)
ax.scatter(
    xy_train[:, 0],
    xy_train[:, 1],
    s=4,
    c="#3a3a3a",
    alpha=0.18,
    linewidths=0,
    label=f"Train tasks (n={len(train_ids)})",
    zorder=2,
)

# 6c. convex-hull tints behind major test families — only the highlighted
# families to keep the figure clean.
HULL_FAMILIES = set(HIGHLIGHT_FAMILIES)
for fam in HULL_FAMILIES:
    color = palette.get(fam, "#888")
    pts = xy_test[families.values == fam]
    # only points inside the cropped view
    inside = (pts[:, 0] >= xlo) & (pts[:, 0] <= xhi) & (pts[:, 1] >= ylo) & (pts[:, 1] <= yhi)
    pts = pts[inside]
    if len(pts) >= 4:
        try:
            hull = ConvexHull(pts)
            poly = plt.Polygon(
                pts[hull.vertices],
                closed=True,
                fill=True,
                facecolor=color,
                alpha=0.07,
                edgecolor=color,
                linewidth=0.4,
                linestyle="--",
                zorder=2.5,
            )
            ax.add_patch(poly)
        except Exception:
            pass

# 6d. test scatter (large, colored)
ax.scatter(
    xy_test[:, 0],
    xy_test[:, 1],
    s=55,
    c=test_colors,
    edgecolors="white",
    linewidths=0.8,
    zorder=4,
    alpha=0.95,
)

# 6e. arrows from highlighted targets to top-3 OTDD-nearest train sources
for ti in target_picks:
    target_xy = xy_test[ti]
    for rank, src_idx in enumerate(nn3_per_target[ti]):
        src_xy = xy_train[src_idx]
        d = otdd_mat[src_idx, ti]
        # alpha: closer sources darker; thicker arrows so they read clearly
        a = float(np.clip(1.0 - rank * 0.22, 0.55, 1.0))
        arrow = FancyArrowPatch(
            (target_xy[0], target_xy[1]),
            (src_xy[0], src_xy[1]),
            arrowstyle="-|>",
            mutation_scale=15,
            color="#1f1f1f",
            alpha=a,
            linewidth=2.0,
            connectionstyle="arc3,rad=0.20",
            zorder=5,
        )
        ax.add_patch(arrow)
        # halo on the source train task
        ax.scatter(
            [src_xy[0]],
            [src_xy[1]],
            s=80,
            facecolors="none",
            edgecolors="#1f1f1f",
            linewidths=1.2,
            alpha=a,
            zorder=4.5,
        )
    # star marker on the target
    ax.scatter(
        [target_xy[0]],
        [target_xy[1]],
        marker="*",
        s=320,
        c="white",
        edgecolors="black",
        linewidths=1.4,
        zorder=6,
    )
    # CHEMBL label — placed in 8 directions cycling around so they don't
    # collide. With MIN_SEP enforced on the targets themselves, even simple
    # offsets work.
    pos = target_picks.index(ti)
    # cycle through compact diagonal offsets to spread labels around the markers
    offsets = [(18, 14), (-100, 14), (18, -22), (-100, -22), (24, 0)]
    label_offset = offsets[pos % len(offsets)]
    ax.annotate(
        f"{test_ids[ti]}\n{fam_arr[ti]}",
        xy=(target_xy[0], target_xy[1]),
        xytext=label_offset,
        textcoords="offset points",
        fontsize=9,
        fontweight="bold",
        color="#111",
        zorder=7,
        bbox=dict(boxstyle="round,pad=0.30", fc="white", ec="#666", linewidth=0.6, alpha=0.96),
        arrowprops=dict(arrowstyle="-", color="#888", linewidth=0.6, connectionstyle="arc3,rad=0"),
        ha="left" if label_offset[0] >= 0 else "left",
    )

# 6f. title — placed in the top-left where the cropped island left empty space
ax.text(
    0.012,
    0.985,
    "A map of medicinal-chemistry task space",
    transform=ax.transAxes,
    fontsize=21,
    fontweight="bold",
    va="top",
    ha="left",
    color="#111",
)
ax.text(
    0.012,
    0.945,
    f"UMAP of {len(train_ids) + len(test_ids):,} FS-Mol task centroids "
    f"(GIN-supervised-infomax)\nArrows: top-3 OTDD-nearest training tasks "
    "for each highlighted target",
    transform=ax.transAxes,
    fontsize=10.5,
    va="top",
    ha="left",
    color="#444",
    style="italic",
)

# legend for families (only those shown)
legend_handles = []
for fam in families.value_counts().index:
    n = (families == fam).sum()
    color = palette.get(fam, "#666")
    legend_handles.append(mpatches.Patch(facecolor=color, edgecolor="white", label=f"{fam} (n={n})"))
legend_handles.append(
    plt.Line2D(
        [],
        [],
        marker="*",
        color="w",
        markerfacecolor="white",
        markeredgecolor="black",
        markeredgewidth=1.2,
        markersize=14,
        label="Highlighted target",
        linestyle="",
    )
)
legend_handles.append(
    plt.Line2D(
        [],
        [],
        color="#222",
        linewidth=1.3,
        marker=">",
        markersize=7,
        label="OTDD top-3 nearest source",
        linestyle="-",
    )
)
ax.legend(
    handles=legend_handles,
    loc="upper right",
    frameon=True,
    framealpha=0.95,
    fontsize=10,
    ncol=1,
    title="Protein family",
    title_fontsize=11,
    edgecolor="#cccccc",
    fancybox=False,
    borderpad=0.8,
)

# Compute Spearman ρ for reference (printed but not plotted)
test_idx_by_id = {t: i for i, t in enumerate(test_ids)}
common_ids = [t for t in test_ids if t in protonet.index]
nn_dist_for_common = np.array([test_nn_dist[test_idx_by_id[t]] for t in common_ids])
delta = protonet.loc[common_ids, "delta_auprc"].values.astype(float)
mask = np.isfinite(nn_dist_for_common) & np.isfinite(delta)
rho, p = spearmanr(nn_dist_for_common[mask], delta[mask])

# Save outputs
out_pdf = ASSETS / "themap_hero_figure.pdf"
out_png = ASSETS / "themap_hero_figure.png"
fig.savefig(out_pdf)
fig.savefig(out_png, dpi=300)
plt.close(fig)

print(f"\nSaved: {out_pdf}  ({out_pdf.stat().st_size / 1024:.0f} KB)")
print(f"Saved: {out_png}  ({out_png.stat().st_size / 1024:.0f} KB)")
print(f"(Validation Spearman ρ = {rho:.3f}, p = {p:.2e}; not shown on map)")